# Fungi exploration

Explore the 12 fungal metabolic models and their accompanying growth data before developing a shared FluxFormer.

- Load the models and inspect their size and compartments.
- Check growth under the supplied medium and test biotinate supplementation.
- Count shared and model-specific reactions.
- List and search all exchange reactions, with chemical identifiers and signed bounds.
- Match experimental CSV chemicals to model exchanges.
- Separate basal ingredients, variable carbon sources, and variable oxygen.
- Test individual carbon sources, including arabinose/lactate sensitivity cases and a no-added-carbon control.

Run cells in order from the repository root. Tests restore the original model bounds. Model predictions describe the tested constraints; they do not by themselves establish biological growth capability.

For experimental growth summaries and medium-composition visualizations, see [fungi_growth_data_exploration.ipynb](fungi_growth_data_exploration.ipynb).

In [1]:
from pathlib import Path
from cobra.io import read_sbml_model

model_dir = Path("metabolic_models")
model_paths = sorted(model_dir.glob("*.sbml"))
if not model_paths:
    raise FileNotFoundError(f"No SBML models found in {model_dir.resolve()}")

print(f"Found {len(model_paths)} SBML models.")

Found 12 SBML models.


In [2]:
models = {}
complex_i_absent_species = ("Hanseniaspora_uvarum_", "Kluyveromyces_marxianus_")

for model_path in model_paths:
    model = read_sbml_model(str(model_path))
    # Sandra's correction: these two yeasts should not contain Complex I.
    if model_path.stem.startswith(complex_i_absent_species):
        model.remove_reactions(["UF02391_M"])
    models[model_path.stem] = model

    print(f"\nFile: {model_path.name}")
    print(f"Model ID: {model.id}")
    print(f"Name: {model.name}")
    print(f"Reactions: {len(model.reactions)}")
    print(f"Metabolites: {len(model.metabolites)}")
    print(f"Genes: {len(model.genes)}")
    print(f"Compartments: {model.compartments}")
    print(f"Exchange reactions: {len(model.exchanges)}")
    print(f"Objective ({model.objective.direction}): {model.objective.expression}")


File: Candida_intermedia_C04520--870.8747989420568-1002.5087414889282.sbml
Model ID: Candida_intermedia_C04520
Name: None
Reactions: 1971
Metabolites: 1456
Genes: 525
Compartments: {'c': 'Cytosol', 'm': 'Mitochondrion', 'r': 'Endoplasmic reticulum', 'e': 'Extracellular', 'x': 'Peroxisome', 'n': 'Nucleus', 'l': 'Lipid particle', 'g': 'Golgi'}
Exchange reactions: 347
Objective (max): 1.0*BIOMASS_C - 1.0*BIOMASS_C_reverse_e6e22

File: Cybelindnera_jadinii_model_kVd1M0R.sbml
Model ID: Universal
Name: None
Reactions: 2164
Metabolites: 1584
Genes: 594
Compartments: {'m': 'Mitochondrion', 'c': 'Cytosol', 'e': 'Extracellular', 'r': 'Endoplasmic reticulum', 'l': 'Lipid particle', 'n': 'Nucleus', 'x': 'Peroxisome', 'g': 'Golgi'}
Exchange reactions: 348
Objective (max): 1.0*BIOMASS_C - 1.0*BIOMASS_C_reverse_e6e22

File: Cyberlindnera_fabianii_C02454--879.9764543360058-1010.5688131565911.sbml
Model ID: Cyberlindnera_fabianii_C02454
Name: None
Reactions: 1818
Metabolites: 1354
Genes: 503
Compartme

## Baseline feasibility under supplied bounds

Run FBA for every model using its loaded bounds and objective. `baseline_results` reports solver status, the biomass objective value only for optimal solutions, and the number of exchanges permitting uptake (from `model.medium`, regardless of actual uptake in the solution).

An optimal solution can have zero biomass. Infeasible means no solution satisfies the constraints; unbounded means the objective has no finite maximum. These are baseline checks, not a ranking of species growth: supplied media, uptake bounds, and biomass definitions may differ. Objective values retain the model's native units.

Solutions are kept in `baseline_solutions`, keyed like `models`; entries are `None` if optimization raises an error. Bounds and objectives are not changed. The loaded models include Sandra's correction: Complex I (`UF02391_M`) is removed from Hanseniaspora uvarum and Kluyveromyces marxianus before any analysis. Original SBML files are preserved.

In [3]:
import pandas as pd
from IPython.display import display
from cobra.exceptions import OptimizationError

baseline_solutions = {}
baseline_rows = []

for model_key, model in models.items():
    error = ""
    try:
        solution = model.optimize()
        status = solution.status
    except OptimizationError as exc:
        solution = None
        status = model.solver.status or "error"
        error = str(exc)

    baseline_solutions[model_key] = solution
    baseline_rows.append({
        "Model file": f"{model_key}.sbml",
        "Solver status": status,
        "Biomass objective value": (
            solution.objective_value
            if solution is not None and status == "optimal"
            else float("nan")
        ),
        "Exchanges permitting uptake": len(model.medium),
        "Error": error,
    })

baseline_results = pd.DataFrame(baseline_rows)
with pd.option_context("display.max_colwidth", None):
    display(baseline_results)

,Model file,Solver status,Biomass objective value,Exchanges permitting uptake,Error
0,Candida_intermedia_C04520--870.8747989420568-1002.5087414889282.sbml,optimal,0.000000,9,
1,Cybelindnera_jadinii_model_kVd1M0R.sbml,optimal,1.143142,10,
2,Cyberlindnera_fabianii_C02454--879.9764543360058-1010.5688131565911.sbml,optimal,0.000000,9,
3,Hanseniaspora_uvarum_C04561--847.6154786717304-1043.5510388250905.sbml,optimal,0.000000,9,
4,Kluyveromyces_marxianus_C191931--863.7832262800852-1014.4827207190165.sbml,optimal,0.000000,9,
5,Meyerozyma_guilliermondii_C04568--842.1876298230761-980.2202031226027.sbml,optimal,0.000000,9,
6,Meyerozyma_guilliermondii_C59004--842.1413028781477-978.6079476256979.sbml,optimal,0.000000,9,
7,Pichia_fermentans_C191029--874.4190878254415-1018.9176350906878.sbml,optimal,0.000000,9,
8,Rhodotorula_badjeave--860.0831274961427-999.2607041864406.sbml,optimal,0.000000,9,
9,Rhodotorula_diarensis_model_g0zrTUR.sbml,optimal,0.932440,10,


## Permitted uptake substances and bounds

List every exchange permitting uptake in each model's current medium. Exchanges are selected using `model.medium`; the table reports their signed lower and upper bounds. For these exchanges, negative flux denotes uptake and positive flux denotes secretion. Bounds constrain allowed rates in the model's native flux units; they are not concentrations or actual consumption. Bounds are not changed.

The combined table is available as `uptake_results`. Compare metabolite identities and bounds across models; matching reaction IDs alone do not guarantee identical substances.

In [4]:
uptake_rows = []

for model_key, model in models.items():
    for reaction_id in sorted(model.medium):
        reaction = model.reactions.get_by_id(reaction_id)
        uptake_rows.append({
            "Model file": f"{model_key}.sbml",
            "Exchange ID": reaction.id,
            "Metabolite IDs": ", ".join(m.id for m in reaction.metabolites),
            "Substances": ", ".join(m.name or m.id for m in reaction.metabolites),
            "Lower bound": reaction.lower_bound,
            "Upper bound": reaction.upper_bound,
        })

uptake_results = pd.DataFrame(uptake_rows, columns=[
    "Model file", "Exchange ID", "Metabolite IDs", "Substances",
    "Lower bound", "Upper bound",
])

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    for model_key in models:
        model_file = f"{model_key}.sbml"
        print(f"\n{model_file}")
        display(uptake_results.loc[
            uptake_results["Model file"] == model_file
        ].drop(columns="Model file").reset_index(drop=True))


Candida_intermedia_C04520--870.8747989420568-1002.5087414889282.sbml


,Exchange ID,Metabolite IDs,Substances,Lower bound,Upper bound
0,UF02549_E,ash_1g_e,ash 1 g,-1000.0,1000.0
1,UF02765_E,pi_e,hydrogenphosphate,-1000.0,1000.0
2,UF03268_E,fe2_e,iron(2+),-1000.0,1000.0
3,UF03288_E,glcD_e,alpha-D-glucose,-10.0,1000.0
4,UF03314_E,h2o_e,water,-1000.0,1000.0
5,UF03376_E,nh4_e,ammonium,-1000.0,1000.0
6,UF03382_E,o2_e,dioxygen,-1000.0,1000.0
7,UF03456_E,so4_e,sulfate,-1000.0,1000.0
8,UF03474_E,h_e,H+,-1000.0,1000.0



Cybelindnera_jadinii_model_kVd1M0R.sbml


,Exchange ID,Metabolite IDs,Substances,Lower bound,Upper bound
0,UF02549_E,ash_1g_e,ash 1 g,-1000.0,1000.0
1,UF02765_E,pi_e,hydrogenphosphate,-1000.0,1000.0
2,UF02997_E,btn_e,biotinate,-1000.0,1000.0
3,UF03268_E,fe2_e,iron(2+),-1000.0,1000.0
4,UF03288_E,glcD_e,alpha-D-glucose,-10.0,1000.0
5,UF03314_E,h2o_e,water,-1000.0,1000.0
6,UF03376_E,nh4_e,ammonium,-1000.0,1000.0
7,UF03382_E,o2_e,dioxygen,-1000.0,1000.0
8,UF03456_E,so4_e,sulfate,-1000.0,1000.0
9,UF03474_E,h_e,H+,-1000.0,1000.0



Cyberlindnera_fabianii_C02454--879.9764543360058-1010.5688131565911.sbml


...

,Exchange ID,Metabolite IDs,Substances,Lower bound,Upper bound
0,UF02549_E,ash_1g_e,ash 1 g,-1000.0,1000.0
1,UF02765_E,pi_e,hydrogenphosphate,-1000.0,1000.0
2,UF02997_E,btn_e,biotinate,-1000.0,1000.0
3,UF03268_E,fe2_e,iron(2+),-1000.0,1000.0
4,UF03288_E,glcD_e,alpha-D-glucose,-10.0,1000.0
5,UF03314_E,h2o_e,water,-1000.0,1000.0
6,UF03376_E,nh4_e,ammonium,-1000.0,1000.0
7,UF03382_E,o2_e,dioxygen,-1000.0,1000.0
8,UF03456_E,so4_e,sulfate,-1000.0,1000.0
9,UF03474_E,h_e,H+,-1000.0,1000.0



Rhodotorula_toruloides_C71034--869.4345828760119-988.420654385849.sbml


,Exchange ID,Metabolite IDs,Substances,Lower bound,Upper bound
0,UF02549_E,ash_1g_e,ash 1 g,-1000.0,1000.0
1,UF02765_E,pi_e,hydrogenphosphate,-1000.0,1000.0
2,UF03268_E,fe2_e,iron(2+),-1000.0,1000.0
3,UF03288_E,glcD_e,alpha-D-glucose,-10.0,1000.0
4,UF03314_E,h2o_e,water,-1000.0,1000.0
5,UF03376_E,nh4_e,ammonium,-1000.0,1000.0
6,UF03382_E,o2_e,dioxygen,-1000.0,1000.0
7,UF03456_E,so4_e,sulfate,-1000.0,1000.0
8,UF03474_E,h_e,H+,-1000.0,1000.0



Wicherhamomyces_anomalous_C74022--860.1334398364828-999.2394182528495.sbml


,Exchange ID,Metabolite IDs,Substances,Lower bound,Upper bound
0,UF02549_E,ash_1g_e,ash 1 g,-1000.0,1000.0
1,UF02765_E,pi_e,hydrogenphosphate,-1000.0,1000.0
2,UF03268_E,fe2_e,iron(2+),-1000.0,1000.0
3,UF03288_E,glcD_e,alpha-D-glucose,-10.0,1000.0
4,UF03314_E,h2o_e,water,-1000.0,1000.0
5,UF03376_E,nh4_e,ammonium,-1000.0,1000.0
6,UF03382_E,o2_e,dioxygen,-1000.0,1000.0
7,UF03456_E,so4_e,sulfate,-1000.0,1000.0
8,UF03474_E,h_e,H+,-1000.0,1000.0


## Biotinate sensitivity

Compare the supplied-medium baseline with FBA after permitting biotinate uptake through `UF02997_E` (`btn_e`). Change only its lower bound to −1000, matching the permissive cap already present in the two growing models. This is a sensitivity test, not a measured vitamin requirement or concentration.

Each model is modified inside a temporary context, which restores its original bounds afterward. `biotinate_results` keeps baseline and supplemented results side by side; `biotinate_solutions` stores supplemented solutions. Biomass values are reported only for optimal solutions. Run the baseline section first, without changing model constraints between sections. The original baseline table and solutions are retained.

In [5]:
biotinate_exchange_id = "UF02997_E"
biotinate_uptake_cap = 1000.0
biotinate_rows = []
biotinate_solutions = {}

for model_key, model in models.items():
    exchange = model.reactions.get_by_id(biotinate_exchange_id)
    if len(exchange.metabolites) != 1 or next(iter(exchange.metabolites.values())) != -1:
        raise ValueError(f"Unexpected biotinate exchange convention in {model_key}")

    original_lower_bound = exchange.lower_bound
    baseline_solution = baseline_solutions[model_key]
    baseline_status = (
        baseline_solution.status if baseline_solution is not None
        else baseline_results.set_index("Model file").loc[f"{model_key}.sbml", "Solver status"]
    )
    with model:
        exchange.lower_bound = -biotinate_uptake_cap
        try:
            solution = model.optimize()
            status = solution.status
        except OptimizationError:
            solution = None
            status = model.solver.status or "error"

    biotinate_solutions[model_key] = solution
    biotinate_rows.append({
        "Model file": f"{model_key}.sbml",
        "Original biotinate lower bound": original_lower_bound,
        "Baseline status": baseline_status,
        "Baseline biomass": (
            baseline_solution.objective_value
            if baseline_solution is not None and baseline_status == "optimal"
            else float("nan")
        ),
        "Supplemented biotinate lower bound": -biotinate_uptake_cap,
        "Supplemented status": status,
        "Supplemented biomass": (
            solution.objective_value
            if solution is not None and status == "optimal"
            else float("nan")
        ),
    })

biotinate_results = pd.DataFrame(biotinate_rows).set_index("Model file")
with pd.option_context("display.max_colwidth", None):
    display(biotinate_results)

,Original biotinate lower bound,Baseline status,Baseline biomass,Supplemented biotinate lower bound,Supplemented status,Supplemented biomass
Model file,,,,,,
Candida_intermedia_C04520--870.8747989420568-1002.5087414889282.sbml,0.0,optimal,0.000000,-1000.0,optimal,1.134379
Cybelindnera_jadinii_model_kVd1M0R.sbml,-1000.0,optimal,1.143142,-1000.0,optimal,1.143142
Cyberlindnera_fabianii_C02454--879.9764543360058-1010.5688131565911.sbml,0.0,optimal,0.000000,-1000.0,optimal,1.136288
Hanseniaspora_uvarum_C04561--847.6154786717304-1043.5510388250905.sbml,0.0,optimal,0.000000,-1000.0,optimal,1.028320
Kluyveromyces_marxianus_C191931--863.7832262800852-1014.4827207190165.sbml,0.0,optimal,0.000000,-1000.0,optimal,0.866986
Meyerozyma_guilliermondii_C04568--842.1876298230761-980.2202031226027.sbml,0.0,optimal,0.000000,-1000.0,optimal,0.928571
Meyerozyma_guilliermondii_C59004--842.1413028781477-978.6079476256979.sbml,0.0,optimal,0.000000,-1000.0,optimal,0.873915
Pichia_fermentans_C191029--874.4190878254415-1018.9176350906878.sbml,0.0,optimal,0.000000,-1000.0,optimal,1.116792
Rhodotorula_badjeave--860.0831274961427-999.2607041864406.sbml,0.0,optimal,0.000000,-1000.0,optimal,0.887343


## Reaction inventory: union, shared reactions, and model presence

Count reactions by their COBRA reaction IDs, preserving the supplied namespace. Each distinct ID is counted once in the **union**, even if it appears in many models. The **intersection** contains IDs present in every model. These counts determine a proposed vocabulary; they do not train a model or establish identical stoichiometry for every shared ID.

`reaction_vocabulary` is the sorted union; `shared_reaction_ids` lists the intersection. `reaction_presence` is a boolean model-by-reaction table. A missing reaction differs from a present reaction with zero flux. The inventory and overlap tables use short labels (`M01`, etc.); the inventory gives their full model keys.

`Reaction name` uses the shortest supplied nonempty label, with trailing `_0`/`_1` sequences and `(?)` markers removed for readability. Enzyme classification numbers are retained. All original name variants remain in `reaction_names_by_id`; model names and reaction IDs are unchanged.

In [6]:
import re

if not models:
    raise ValueError("Load the SBML models before building the reaction inventory.")

model_keys = sorted(models)
model_to_index = {key: i for i, key in enumerate(model_keys)}
model_labels = {key: f"M{i + 1:02d}" for i, key in enumerate(model_keys)}
reaction_ids_by_model = {
    key: {reaction.id for reaction in models[key].reactions}
    for key in model_keys
}

all_reaction_ids = set.union(*reaction_ids_by_model.values())
shared_reaction_ids = sorted(set.intersection(*reaction_ids_by_model.values()))
reaction_vocabulary = sorted(all_reaction_ids)
reaction_to_token = {reaction_id: i for i, reaction_id in enumerate(reaction_vocabulary)}
reaction_presence = pd.DataFrame(
    [[reaction_id in reaction_ids_by_model[key] for reaction_id in reaction_vocabulary]
     for key in model_keys],
    index=pd.Index(model_keys, name="Model key"),
    columns=pd.Index(reaction_vocabulary, name="Reaction ID"),
    dtype=bool,
)
reaction_model_counts = reaction_presence.sum(axis=0)
unique_reaction_ids_by_model = {
    key: sorted(reaction_id for reaction_id in reaction_ids_by_model[key]
                if reaction_model_counts[reaction_id] == 1)
    for key in model_keys
}

reaction_inventory = pd.DataFrame([
    {
        "Model label": model_labels[key],
        "Model key": key,
        "Reactions": len(reaction_ids_by_model[key]),
        "Shared by all models": len(shared_reaction_ids),
        "Not shared by all": len(reaction_ids_by_model[key]) - len(shared_reaction_ids),
        "Unique to this model": len(unique_reaction_ids_by_model[key]),
        "Absent from this model": len(reaction_vocabulary) - len(reaction_ids_by_model[key]),
    }
    for key in model_keys
])
reaction_names_by_id = {reaction_id: set() for reaction_id in reaction_vocabulary}
for model in models.values():
    for reaction in model.reactions:
        if reaction.name:
            reaction_names_by_id[reaction.id].add(reaction.name)

def readable_reaction_name(name):
    name = re.sub(r"(?:_[01])+$", "", name)
    name = name.replace("(?)", "")
    return " ".join(name.split())


reaction_catalog = pd.DataFrame({
    "Token index": range(len(reaction_vocabulary)),
    "Reaction ID": reaction_vocabulary,
    "Reaction name": [
        readable_reaction_name(
            min(reaction_names_by_id[reaction_id], key=lambda name: (len(name), name), default="")
        )
        for reaction_id in reaction_vocabulary
    ],
    "Models containing reaction": reaction_model_counts.reindex(reaction_vocabulary).to_numpy(),
}).set_index("Token index")

print(f"Models: {len(model_keys)}")
print(f"Reaction occurrences summed across models: {sum(map(len, reaction_ids_by_model.values()))}")
print(f"Distinct reaction IDs (union): {len(reaction_vocabulary)}")
print(f"Reaction IDs shared by every model (intersection): {len(shared_reaction_ids)}")
print(f"Reaction IDs present in only some models: {len(reaction_vocabulary) - len(shared_reaction_ids)}")
print(f"Reaction-presence table shape: {reaction_presence.shape}")
with pd.option_context("display.max_colwidth", None):
    display(reaction_inventory)
print("First 10 vocabulary entries; the full list is in reaction_catalog:")
with pd.option_context("display.max_colwidth", None):
    display(reaction_catalog.head(10))

Models: 12
Reaction occurrences summed across models: 23736
Distinct reaction IDs (union): 3338
Reaction IDs shared by every model (intersection): 928
Reaction IDs present in only some models: 2410
Reaction-presence table shape: (12, 3338)


,Model label,Model key,Reactions,Shared by all models,Not shared by all,Unique to this model,Absent from this model
0,M01,Candida_intermedia_C04520--870.8747989420568-1002.5087414889282,1971,928,1043,16,1367
1,M02,Cybelindnera_jadinii_model_kVd1M0R,2164,928,1236,57,1174
2,M03,Cyberlindnera_fabianii_C02454--879.9764543360058-1010.5688131565911,1818,928,890,3,1520
3,M04,Hanseniaspora_uvarum_C04561--847.6154786717304-1043.5510388250905,1608,928,680,29,1730
4,M05,Kluyveromyces_marxianus_C191931--863.7832262800852-1014.4827207190165,1751,928,823,6,1587
5,M06,Meyerozyma_guilliermondii_C04568--842.1876298230761-980.2202031226027,1765,928,837,1,1573
6,M07,Meyerozyma_guilliermondii_C59004--842.1413028781477-978.6079476256979,1821,928,893,4,1517
7,M08,Pichia_fermentans_C191029--874.4190878254415-1018.9176350906878,1803,928,875,6,1535
8,M09,Rhodotorula_badjeave--860.0831274961427-999.2607041864406,2273,928,1345,36,1065
9,M10,Rhodotorula_diarensis_model_g0zrTUR,2596,928,1668,256,742


First 10 vocabulary entries; the full list is in reaction_catalog:


,Reaction ID,Reaction name,Models containing reaction
Token index,,,
0,BIOMASS_C,BIOMASS - artificial,12
1,UF00002_C,aldehyde dehydrogenase (1.1.1.-),2
2,UF00003_C,alcohol dehydrogenase (1.1.1.-),4
3,UF00004_C,alcohol dehydrogenase (1.1.1.-),8
4,UF00005_C,alcohol dehydrogenase (1.1.1.-),3
5,UF00006_C,alcohol dehydrogenase (1.1.1.-),4
6,UF00011_C,hydroxymethylfurfural reductase (1.1.1.-),11
7,UF00014_C,primary alcohol:NAD+ oxidoreductase (1.1.1.1),12
8,UF00014_M,primary alcohol:NAD+ oxidoreductase (1.1.1.1),7


### Pairwise overlap and reaction coverage

The overlap table counts shared reaction IDs for every pair of models. Its diagonal equals each model's reaction count. The coverage table shows how many distinct reaction IDs occur in exactly 1, 2, ..., all models.

Inspect the full shared list with `shared_reaction_ids`, a model's unique list with `unique_reaction_ids_by_model[model_key]`, or its presence mask with `reaction_presence.loc[model_key]`. Token and model index mappings are exploratory proposals and must be saved with future datasets/checkpoints.

In [7]:
reaction_overlap = pd.DataFrame(
    [[len(reaction_ids_by_model[left] & reaction_ids_by_model[right])
      for right in model_keys]
     for left in model_keys],
    index=pd.Index([model_labels[key] for key in model_keys], name="Model"),
    columns=[model_labels[key] for key in model_keys],
)
reaction_coverage = (
    reaction_model_counts.value_counts()
    .reindex(range(1, len(model_keys) + 1), fill_value=0)
    .sort_index()
    .rename_axis("Number of models containing reaction")
    .rename("Distinct reaction IDs")
    .to_frame()
)

display(reaction_overlap)
display(reaction_coverage)

,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12
Model,,,,,,,,,,,,
M01,1971,1718,1619,1404,1603,1610,1664,1618,1623,1626,1660,1711
M02,1718,2164,1629,1354,1559,1534,1588,1595,1730,1858,1752,1666
M03,1619,1629,1818,1381,1536,1513,1567,1557,1553,1515,1568,1613
M04,1404,1354,1381,1608,1373,1309,1328,1366,1363,1315,1359,1344
M05,1603,1559,1536,1373,1751,1490,1527,1655,1479,1436,1490,1518
M06,1610,1534,1513,1309,1490,1765,1676,1498,1480,1462,1522,1537
M07,1664,1588,1567,1328,1527,1676,1821,1550,1533,1522,1573,1582
M08,1618,1595,1557,1366,1655,1498,1550,1803,1514,1477,1546,1569
M09,1623,1730,1553,1363,1479,1480,1533,1514,2273,2027,2071,1585


,Distinct reaction IDs
Number of models containing reaction,
1,446
2,250
3,359
4,240
5,125
6,130
7,116
8,104
9,151


## Complete exchange-reaction catalogue

List every distinct exchange, including those closed to uptake in the supplied models. The compact table shows exchange ID, metabolite name/ID, formula, and model count. All rows remain available in a scrollable table; long text wraps instead of stretching columns.

Chemical identifiers are **database references for the molecule**, not reaction IDs, fluxes, or measurements. For example, `ChEBI: CHEBI:17108`, `KEGG Compound: C00216`, and `PubChem: 351` are references supplied for D-arabinose. They help verify chemical identity when names differ. They are shown separately for search matches, keeping the main catalogue narrow.

`exchange_catalog` retains all metadata; `exchange_identifiers` separates each database/identifier pair and counts models providing that annotation. Missing annotations do not mean missing compounds. Names/formulas come from SBML; experimental CSV names appear in the later mapping tables.

In [8]:
from IPython.display import HTML


def display_exchange_table(frame):
    table = frame.reset_index() if any(name is not None for name in frame.index.names) else frame
    html = table.to_html(index=False, border=0, escape=True)
    display(HTML(
        "<style>"
        ".fungi-table {display:inline-block; max-width:100%; max-height:34rem; overflow:auto;}"
        ".fungi-table table {border-collapse:collapse; width:auto; font-size:0.9em;}"
        ".fungi-table th, .fungi-table td {padding:4px 8px; text-align:left; "
        "white-space:normal; overflow-wrap:anywhere; max-width:22rem;}"
        ".fungi-table thead th {white-space:nowrap;}"
        ".fungi-table thead {position:sticky; top:0; background:#f5f5f5; color:#222;}"
        "</style><div class='fungi-table'>" + html + "</div>"
    ))


def metabolite_identifiers(metabolite):
    identifiers = []
    for namespace, values in sorted(metabolite.annotation.items()):
        if namespace == "sbo":
            continue
        if isinstance(values, str):
            values = [values]
        identifiers.extend(f"{namespace}: {value}" for value in values)
    return "; ".join(sorted(set(identifiers))) or "Not provided"


exchange_detail_rows = []
exchange_identifier_rows = []
database_labels = {"chebi": "ChEBI", "kegg.compound": "KEGG Compound", "pubchem.compound": "PubChem"}
for model_key in model_keys:
    for reaction in models[model_key].exchanges:
        metabolites = list(reaction.metabolites)
        exchange_detail_rows.append({
            "Exchange ID": reaction.id,
            "Model label": model_labels[model_key],
            "Model key": model_key,
            "Reaction name": readable_reaction_name(reaction.name or reaction.id),
            "Metabolite IDs": ", ".join(sorted(m.id for m in metabolites)),
            "Model metabolite names": ", ".join(sorted(m.name or m.id for m in metabolites)),
            "Formula": ", ".join(sorted({m.formula or "Not provided" for m in metabolites})),
            "Chemical identifiers": " | ".join(sorted({metabolite_identifiers(m) for m in metabolites})),
            "Lower bound": reaction.lower_bound,
            "Upper bound": reaction.upper_bound,
        })
        for metabolite in metabolites:
            for namespace, values in metabolite.annotation.items():
                if namespace == "sbo":
                    continue
                if isinstance(values, str):
                    values = [values]
                for value in values:
                    identifier = str(value)
                    if namespace == "chebi":
                        identifier = re.sub(r"^chebi:", "CHEBI:", identifier, flags=re.IGNORECASE)
                    exchange_identifier_rows.append({
                        "Exchange ID": reaction.id,
                        "Database": database_labels.get(namespace, namespace),
                        "Identifier": identifier,
                        "Model key": model_key,
                    })
exchange_details = pd.DataFrame(exchange_detail_rows)
exchange_identifiers = (
    pd.DataFrame(exchange_identifier_rows, columns=["Exchange ID", "Database", "Identifier", "Model key"])
    .groupby(["Exchange ID", "Database", "Identifier"], sort=True)
    .agg(Models=("Model key", "nunique"))
)
identifier_labels = exchange_identifiers.reset_index()
identifier_labels["Reference"] = identifier_labels["Database"] + ": " + identifier_labels["Identifier"]
identifier_summary = identifier_labels.groupby("Exchange ID")["Reference"].agg("; ".join)
exchange_catalog = exchange_details.groupby("Exchange ID", sort=True).agg(
    **{
        column: (column, lambda values: " / ".join(sorted(set(values))))
        for column in ["Reaction name", "Metabolite IDs", "Model metabolite names", "Formula"]
    },
    **{"Models containing exchange": ("Model key", "nunique")},
)
exchange_catalog["Chemical identifiers"] = identifier_summary.reindex(exchange_catalog.index).fillna("Not provided")
exchange_presence = (
    exchange_details.assign(Present=1)
    .pivot(index="Exchange ID", columns="Model label", values="Present")
    .reindex(columns=[model_labels[key] for key in model_keys])
    .fillna(0).astype(int)
)
exchange_bounds = (
    exchange_details.set_index(["Model label", "Exchange ID"])
    [["Model key", "Model metabolite names", "Lower bound", "Upper bound"]]
    .sort_index()
)
exchange_display_columns = ["Model metabolite names", "Metabolite IDs", "Formula", "Models containing exchange"]
exchange_display_labels = {"Model metabolite names": "Name", "Metabolite IDs": "Metabolite ID", "Models containing exchange": "Models"}

print(f"Distinct exchange IDs across all models: {len(exchange_catalog)}")
print(f"Exchanges present in all {len(model_keys)} models: {exchange_catalog['Models containing exchange'].eq(len(model_keys)).sum()}")
display_exchange_table(exchange_catalog[exchange_display_columns].rename(columns=exchange_display_labels))

Distinct exchange IDs across all models: 461
Exchanges present in all 12 models: 129


Exchange ID,Name,Metabolite ID,Formula,Models
UF01044_E,O-acetyl-L-carnitine,acrn_e,C9H17NO4,11
UF02476_E,(1->3)-beta-D-glucan,13BDglcn_e,C6H10O5,12
UF02481_E,(2-hydroxyphenyl)acetate,2hyoxplac_e,C8H7O3,10
UF02483_E,2-methylbutyrate,2mba_e,C5H9O2,11
UF02510_E,(4-hydroxyphenyl)acetate,4hoxpac_e,C8H7O3,12
UF02516_E,8-amino-7-oxonooic acid zwitterion,8aonn_e,C9H17NO3,9
UF02529_E,adenine,ade_e,C5H5N5,12
UF02530_E,adenosine,adn_e,C10H13N5O4,12
UF02535_E,beta-alanine zwitterion,ala_B_e,C3H7NO2,12
UF02548_E,L-argininium,argL_e,C6H15N4O2,12


### Exchange presence by model

`exchange_presence` shows all exchanges: 1 = present, 0 = absent. Model labels refer to the full filenames in `reaction_inventory`. Presence does not mean uptake is enabled.

Exact signed bounds remain available in `exchange_bounds` for inspection when needed.

In [9]:
print("Exchange presence by model:")
exchange_presence_display = exchange_presence.copy()
exchange_presence_display.insert(0, "Name", exchange_catalog["Model metabolite names"])
display_exchange_table(exchange_presence_display)

Exchange presence by model:


Model label,Exchange ID,Name,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12
,UF01044_E,O-acetyl-L-carnitine,1,1,1,0,1,1,1,1,1,1,1,1
,UF02476_E,(1->3)-beta-D-glucan,1,1,1,1,1,1,1,1,1,1,1,1
,UF02481_E,(2-hydroxyphenyl)acetate,1,1,1,1,0,1,1,0,1,1,1,1
,UF02483_E,2-methylbutyrate,1,1,1,0,1,1,1,1,1,1,1,1
,UF02510_E,(4-hydroxyphenyl)acetate,1,1,1,1,1,1,1,1,1,1,1,1
,UF02516_E,8-amino-7-oxonooic acid zwitterion,1,1,1,1,0,0,0,1,1,1,1,1
,UF02529_E,adenine,1,1,1,1,1,1,1,1,1,1,1,1
,UF02530_E,adenosine,1,1,1,1,1,1,1,1,1,1,1,1
,UF02535_E,beta-alanine zwitterion,1,1,1,1,1,1,1,1,1,1,1,1
,UF02548_E,L-argininium,1,1,1,1,1,1,1,1,1,1,1,1


### Search the complete exchange catalogue

Change `exchange_search_term` to a name fragment, reaction/metabolite ID, formula, or chemical identifier, such as `arabinose`, `acetate`, or a ChEBI ID. Search is case-insensitive and literal. An empty term selects all exchanges.

The first table shows matching compounds. The second lists their database references, with one row per identifier; `Models` counts models supplying that reference. References missing in some models are not repeated as `Not provided` alongside available IDs. Text matches remain candidates for inspection, not proof of chemical identity.

In [10]:
exchange_search_term = "arabinose"
search_columns = ["Reaction name", "Metabolite IDs", "Model metabolite names", "Formula", "Chemical identifiers"]
searchable_exchanges = exchange_catalog.reset_index()
if exchange_search_term.strip():
    matched_rows = searchable_exchanges[["Exchange ID", *search_columns]].apply(
        lambda column: column.astype(str).str.contains(exchange_search_term.strip(), case=False, regex=False)
    ).any(axis=1)
    exchange_search_results = searchable_exchanges.loc[matched_rows].set_index("Exchange ID")
else:
    exchange_search_results = exchange_catalog.copy()

print(f"{len(exchange_search_results)} matching exchange IDs for {exchange_search_term!r}")
display_exchange_table(exchange_search_results[exchange_display_columns].rename(columns=exchange_display_labels))
print("Database references for matching exchanges (not fluxes or measurements):")
matched_identifiers = exchange_identifiers.loc[
    exchange_identifiers.index.get_level_values("Exchange ID").isin(exchange_search_results.index)
]
display_exchange_table(matched_identifiers)


2 matching exchange IDs for 'arabinose'


Exchange ID,Name,Metabolite ID,Formula,Models
UF03192_E,D-arabinose,arabD_e,C5H10O5,12
UF04276_E,D-arabinose 5-phosphate,arabinophospha_e,C5H9O8P,10


Database references for matching exchanges (not fluxes or measurements):


Exchange ID,Database,Identifier,Models
UF03192_E,ChEBI,CHEBI:17108,10
UF03192_E,KEGG Compound,C00216,10
UF03192_E,PubChem,351,10
UF04276_E,ChEBI,CHEBI:57693,9
UF04276_E,PubChem,9936487,9


## Experimental chemicals and exchange coverage

Read the 25 chemical fields from the supplied growth CSV. Count positive values across the **29 distinct media**, not the repeated organism rows. The registry below records candidate extracellular metabolite identities explicitly; exchange IDs and model coverage are then obtained from the loaded models.

Chemical matching is separate from growth support. An exchange's presence does not establish uptake, catabolism, or inhibition. Stereochemistry, ambiguous positional isomers, and source-column spellings still need confirmation. In particular, the CSV's `HMF` must **not** be mapped to `hmf_e`: that model metabolite is named 5-hydroxymethyluracil.

Chemical units are unconfirmed. The inventory does not turn concentrations into flux bounds or change any model.

In the tables, **Experimental CSV column** is the exact original column name from `Growth_data_for_ML_Helsinki_Uni.csv`, including its spelling and capitalization. **Model metabolite names** comes from the SBML models; it is not a renamed CSV field. **Candidate exchange IDs** comes from model reactions. Compare these side by side to see the proposed mapping; `Not identified` leaves the CSV field visible without inventing a model match.

In [11]:
experimental_data = pd.read_csv(model_dir / "Growth_data_for_ML_Helsinki_Uni.csv")
nonchemical_fields = {
    "organism_id", "medium_id", "AUC", "NetAUC_sandra", "AUC_sandra",
    "growth_max", "growth_max_sandra", "Biomass", "Net_Biomass",
    "Relative_control", "Temperature", "Time", "Dilution",
}
chemical_fields = [column for column in experimental_data if column not in nonchemical_fields]

# Candidate identities refer to the supplied metabolite namespace, not fuzzy ID matches.
chemical_registry = {
    "arabinose": (("arabD_e",), "D-arabinose candidate; CSV stereochemistry unspecified"),
    "glucose": (("glcD_e",), "alpha-D-glucose candidate"),
    "galactose": (("gal_e",), "D-galactopyranose candidate"),
    "xylose": (("xylD_e",), "D-xylose candidate"),
    "manose": (("man_e",), "User confirmed CSV manose means mannose; model form is D-mannose"),
    "fructose": (("fru_e",), "D-fructose candidate"),
    "lactate": (("lacL_e", "lacD_e"), "Both L/(S) and D/(R) forms; do not count one measurement twice"),
    "acetate": (("ac_e",), "Acetate exchange"),
    "benzoic_acid": (("bz_e",), "Benzoate candidate; role as nutrient/inhibitor unresolved"),
    "3-hydroxypropionic": ((), "No matching exchange identified"),
    "sinapyl_alcohol": ((), "No matching exchange identified"),
    "furfural": ((), "No matching exchange identified"),
    "HMF": ((), "No matching exchange identified; hmf_e is hydroxymethyluracil"),
    "chorogenic_acid": ((), "Confirm intended chemical spelling; no matching exchange identified"),
    "gallic_acid": ((), "No matching exchange identified"),
    "sinapic_acid": ((), "No matching exchange identified"),
    "Syringic_acid": ((), "No matching exchange identified"),
    "Hydroxybenzoic_acid": (("4hbz_e",), "4-hydroxybenzoate candidate; CSV positional isomer unspecified"),
    "Ellagic_acid": ((), "No matching exchange identified"),
    "Protocatehuic_acid": (("34dhbz_e",), "3,4-dihydroxybenzoate candidate; confirm intended CSV identity"),
    "Caffeic_acid": ((), "trans-caffeate exists internally (34dhcinm_c); no matching exchange identified"),
    "Ferulic_acid": ((), "Ferulate exists internally (fer_c); no matching exchange identified"),
    "Cinnamic_acid": ((), "trans-cinnamate exists internally (cinnm_c); no matching exchange identified"),
    "Coumaric_acid": (("T4hcinnm_e",), "trans-4-coumarate candidate; CSV isomer unspecified"),
    "Vanillic_acid": (("vanlt_e",), "Vanillate candidate; role as nutrient/inhibitor unresolved"),
}
if set(chemical_fields) != set(chemical_registry):
    raise ValueError("CSV chemical fields differ from the reviewed registry; review mappings.")
if experimental_data.groupby("medium_id")[chemical_fields].nunique(dropna=False).gt(1).any().any():
    raise ValueError("Chemical values differ across organisms for the same medium.")
experimental_media = experimental_data.drop_duplicates("medium_id").set_index("medium_id")

exchanges_by_model = {
    key: {reaction.id: reaction for reaction in model.exchanges}
    for key, model in models.items()
}
exchange_ids_by_metabolite = {}
metabolite_names_by_id = {}
for exchanges in exchanges_by_model.values():
    for reaction in exchanges.values():
        for metabolite in reaction.metabolites:
            exchange_ids_by_metabolite.setdefault(metabolite.id, set()).add(reaction.id)
            metabolite_names_by_id.setdefault(metabolite.id, set()).add(metabolite.name or metabolite.id)

carbon_fields = ["arabinose", "glucose", "galactose", "xylose", "manose", "fructose", "lactate", "acetate"]
chemical_exchange_ids = {}
chemical_inventory_rows = []
for field in chemical_fields:
    metabolite_ids, note = chemical_registry[field]
    exchange_ids = sorted({
        reaction_id for metabolite_id in metabolite_ids
        for reaction_id in exchange_ids_by_metabolite.get(metabolite_id, set())
    })
    chemical_exchange_ids[field] = exchange_ids
    chemical_inventory_rows.append({
        "Experimental CSV column": field,
        "Model metabolite names": ", ".join(sorted({
            name for metabolite_id in metabolite_ids
            for name in metabolite_names_by_id.get(metabolite_id, set())
        })) or "Not identified",
        "Input group": "Variable carbon candidate" if field in carbon_fields else "Other chemical; role unresolved",
        "Candidate exchange IDs": ", ".join(exchange_ids) or "Not identified",
        "Models with any candidate exchange": sum(
            any(reaction_id in exchanges_by_model[key] for reaction_id in exchange_ids)
            for key in model_keys
        ),
        "Media with positive value": int(experimental_media[field].gt(0).sum()),
        "Mapping note": note,
    })
chemical_inventory = pd.DataFrame(chemical_inventory_rows).set_index("Experimental CSV column")
print(f"{len(chemical_fields)} chemical fields across {len(experimental_media)} distinct media")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(chemical_inventory)

25 chemical fields across 29 distinct media


,Model metabolite names,Input group,Candidate exchange IDs,Models with any candidate exchange,Media with positive value,Mapping note
Experimental CSV column,,,,,,
arabinose,D-arabinose,Variable carbon candidate,UF03192_E,12,19,D-arabinose candidate; CSV stereochemistry unspecified
glucose,alpha-D-glucose,Variable carbon candidate,UF03288_E,12,29,alpha-D-glucose candidate
galactose,D-galactopyranose,Variable carbon candidate,UF03281_E,7,19,D-galactopyranose candidate
xylose,D-xylose,Variable carbon candidate,UF03503_E,9,23,D-xylose candidate
manose,D-mannose,Variable carbon candidate,UF03359_E,12,12,User confirmed CSV manose means mannose; model form is D-mannose
fructose,D-fructose,Variable carbon candidate,UF03275_E,12,16,D-fructose candidate
lactate,"(R)-lactate, (S)-lactate",Variable carbon candidate,"UF02967_E, UF02971_E",12,13,Both L/(S) and D/(R) forms; do not count one measurement twice
acetate,acetate,Variable carbon candidate,UF03160_E,11,13,Acetate exchange
benzoic_acid,benzoate(1-),Other chemical; role unresolved,UF03209_E,12,23,Benzoate candidate; role as nutrient/inhibitor unresolved


### Candidate exchange names and presence by model

The first table places each exact experimental CSV column name alongside its candidate exchange ID and the metabolite names from the SBML models. The second shows exactly which models contain it: 1 = present, 0 = absent. Model labels are the same as in `reaction_inventory`. Two lactate rows are alternative stereochemical candidates, not two independently measured nutrients. The tables remain available as `chemical_exchange_catalog` and `chemical_exchange_presence`.

In [12]:
chemical_exchange_rows = []
chemical_presence_rows = []
for field, exchange_ids in chemical_exchange_ids.items():
    for reaction_id in exchange_ids:
        reactions = [exchanges_by_model[key][reaction_id] for key in model_keys
                     if reaction_id in exchanges_by_model[key]]
        metabolite_ids = sorted({m.id for reaction in reactions for m in reaction.metabolites})
        chemical_exchange_rows.append({
            "Experimental CSV column": field,
            "Exchange ID": reaction_id,
            "Metabolite IDs": ", ".join(metabolite_ids),
            "Model metabolite names": ", ".join(sorted({name for mid in metabolite_ids
                                           for name in metabolite_names_by_id[mid]})),
            "Models containing exchange": len(reactions),
        })
        chemical_presence_rows.append({
            "Experimental CSV column": field,
            "Exchange ID": reaction_id,
            **{model_labels[key]: int(reaction_id in exchanges_by_model[key]) for key in model_keys},
        })
chemical_exchange_catalog = pd.DataFrame(chemical_exchange_rows).set_index(["Experimental CSV column", "Exchange ID"])
chemical_exchange_presence = pd.DataFrame(chemical_presence_rows).set_index(["Experimental CSV column", "Exchange ID"])
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(chemical_exchange_catalog)
    display(chemical_exchange_presence)

Metabolite IDs Model metabolite names  \
Experimental CSV column Exchange ID                                         
arabinose               UF03192_E          arabD_e            D-arabinose   
glucose                 UF03288_E           glcD_e        alpha-D-glucose   
galactose               UF03281_E            gal_e      D-galactopyranose   
xylose                  UF03503_E           xylD_e               D-xylose   
manose                  UF03359_E            man_e              D-mannose   
fructose                UF03275_E            fru_e             D-fructose   
lactate                 UF02967_E           lacL_e            (S)-lactate   
                        UF02971_E           lacD_e            (R)-lactate   
acetate                 UF03160_E             ac_e                acetate   
benzoic_acid            UF03209_E             bz_e           benzoate(1-)   
Hydroxybenzoic_acid     UF03142_E           4hbz_e      4-hydroxybenzoate   
Protocatehuic_acid      UF03118_E         34dhbz_e  3,4-dihydroxybenzoate   
Coumaric_acid           UF03565_E       T4hcinnm_e  trans-4-coumarate(1-)   
Vanillic_acid           UF03497_E          vanlt_e              vanillate   

                                     Models containing exchange  
Experimental CSV column Exchange ID                              
arabinose               UF03192_E                            12  
glucose                 UF03288_E                            12  
galactose               UF03281_E                             7  
xylose                  UF03503_E                             9  
manose                  UF03359_E                            12  
fructose                UF03275_E                            12  
lactate                 UF02967_E                            12  
                        UF02971_E                            12  
acetate                 UF03160_E                            11  
benzoic_acid            UF03209_E                            12  
Hydroxybenzoic_acid     UF03142_E                             8  
Protocatehuic_acid      UF03118_E                             4  
Coumaric_acid           UF03565_E                             2  
Vanillic_acid           UF03497_E                             8

M01  M02  M03  M04  M05  M06  M07  M08  \
Experimental CSV column Exchange ID                                           
arabinose               UF03192_E      1    1    1    1    1    1    1    1   
glucose                 UF03288_E      1    1    1    1    1    1    1    1   
galactose               UF03281_E      1    0    1    1    1    1    1    0   
xylose                  UF03503_E      1    1    1    1    1    1    1    1   
manose                  UF03359_E      1    1    1    1    1    1    1    1   
fructose                UF03275_E      1    1    1    1    1    1    1    1   
lactate                 UF02967_E      1    1    1    1    1    1    1    1   
                        UF02971_E      1    1    1    1    1    1    1    1   
acetate                 UF03160_E      1    1    1    1    1    0    1    1   
benzoic_acid            UF03209_E      1    1    1    1    1    1    1    1   
Hydroxybenzoic_acid     UF03142_E      0    1    1    0    1    0    0    1   
Protocatehuic_acid      UF03118_E      1    0    0    1    1    0    0    1   
Coumaric_acid           UF03565_E      1    0    0    0    0    0    0    0   
Vanillic_acid           UF03497_E      1    1    0    0    0    1    1    0   

                                     M09  M10  M11  M12  
Experimental CSV column Exchange ID                      
arabinose               UF03192_E      1    1    1    1  
glucose                 UF03288_E      1    1    1    1  
galactose               UF03281_E      0    1    0    0  
xylose                  UF03503_E      0    0    0    1  
manose                  UF03359_E      1    1    1    1  
fructose                UF03275_E      1    1    1    1  
lactate                 UF02967_E      1    1    1    1  
                        UF02971_E      1    1    1    1  
acetate                 UF03160_E      1    1    1    1  
benzoic_acid            UF03209_E      1    1    1    1  
Hydroxybenzoic_acid     UF03142_E      1    1    1    1  
Protocatehuic_acid      UF03118_E      0    0    0    0  
Coumaric_acid           UF03565_E      0    0    0    1  
Vanillic_acid           UF03497_E      1    1    1    1

### Variable carbon sources and oxygen

Treat the six sugar fields, lactate, and acetate as **candidate variable carbon channels**, subject to subsequent uptake/growth tests. Oxygen is a **separate variable input**, not a fixed basal ingredient and not part of the carbon-source count. It is absent from the chemical CSV, so its sampled bounds must be documented separately. No oxygen range is selected here.

Other experimental chemicals remain separate until their roles and model support are resolved. Missing exchanges are not silently added. This inventory does not prove that all candidate carbon sources support growth in every model.

In [13]:
variable_input_rows = [
    {
        "Input": field,
        "Role": "Variable carbon candidate",
        "Candidate exchange IDs": ", ".join(chemical_exchange_ids[field]) or "Not identified",
        "Models with any candidate exchange": chemical_inventory.loc[field, "Models with any candidate exchange"],
        "Measured in CSV": True,
    }
    for field in carbon_fields
]
oxygen_exchange_id = "UF03382_E"
variable_input_rows.append({
    "Input": "oxygen",
    "Role": "Variable oxygen",
    "Candidate exchange IDs": oxygen_exchange_id,
    "Models with any candidate exchange": sum(oxygen_exchange_id in exchanges for exchanges in exchanges_by_model.values()),
    "Measured in CSV": False,
})
variable_input_inventory = pd.DataFrame(variable_input_rows).set_index("Input")
with pd.option_context("display.max_colwidth", None):
    display(variable_input_inventory)

,Role,Candidate exchange IDs,Models with any candidate exchange,Measured in CSV
Input,,,,
arabinose,Variable carbon candidate,UF03192_E,12,True
glucose,Variable carbon candidate,UF03288_E,12,True
galactose,Variable carbon candidate,UF03281_E,7,True
xylose,Variable carbon candidate,UF03503_E,9,True
manose,Variable carbon candidate,UF03359_E,12,True
fructose,Variable carbon candidate,UF03275_E,12,True
lactate,Variable carbon candidate,"UF02967_E, UF02971_E",12,True
acetate,Variable carbon candidate,UF03160_E,11,True
oxygen,Variable oxygen,UF03382_E,12,False


### Recorded carbon-source composition of the 29 experimental media

Show each medium once, since the same recipe values repeat across all 12 organism groups. List positive values among the eight candidate carbon fields: six sugars, lactate, and acetate. Display `manose` as the user-confirmed name **mannose**; retain the exact CSV column name in the underlying values table.

These are **recorded candidate carbon compounds**, not proof that a model consumes them or a complete medium recipe. Other measured chemicals may have metabolic roles, and unlisted ingredients in YPD/YNB are not assumed absent. A row listing only glucose means only glucose among these eight CSV fields, not necessarily glucose as the only carbon-containing ingredient in the real medium.

`experimental_carbon_composition` lists all 29 profiles. `carbon_composition_values` retains their numeric CSV values (units unconfirmed); inspect a medium with `carbon_composition_values.loc[medium_name]`. No concentration is converted to a flux bound here.

In [14]:
carbon_composition_values = experimental_media[carbon_fields].copy()
recorded_carbon_presence = carbon_composition_values.gt(0)
carbon_display_names = {field: "mannose" if field == "manose" else field for field in carbon_fields}
experimental_carbon_composition = pd.DataFrame({
    "Medium": carbon_composition_values.index,
    "Recorded candidate carbon compounds": [
        ", ".join(carbon_display_names[field] for field in carbon_fields if row[field])
        for _, row in recorded_carbon_presence.iterrows()
    ],
    "Count": recorded_carbon_presence.sum(axis=1).to_numpy(),
}).set_index("Medium")

print(f"Distinct experimental media: {len(experimental_carbon_composition)}")
print(f"Media recording glucose: {int(recorded_carbon_presence['glucose'].sum())}")
print(f"One recorded candidate carbon compound: {int(experimental_carbon_composition['Count'].eq(1).sum())}; mixtures: {int(experimental_carbon_composition['Count'].gt(1).sum())}")
display_exchange_table(experimental_carbon_composition)

Distinct experimental media: 29
Media recording glucose: 29
One recorded candidate carbon compound: 5; mixtures: 24


Medium,Recorded candidate carbon compounds,Count
"Medium 1 (wheat straw 170, 10)","arabinose, glucose, galactose, xylose, fructose, acetate",6
"Medium 2(wheat straw 190, 10)","arabinose, glucose, galactose, xylose, fructose, acetate",6
"Medium 3 (wheat straw 200, 10)","arabinose, glucose, galactose, xylose, fructose, acetate",6
"Medium 4(wheat straw 200, 10 / 2x)","arabinose, glucose, galactose, xylose, fructose, acetate",6
Medium 5(oat straw 170 10),"arabinose, glucose, galactose, xylose, lactate, acetate",6
Medium 6(oat straw 190 10),"arabinose, glucose, galactose, xylose, lactate, acetate",6
"Medium 7(oat straw 200, 10)","arabinose, glucose, galactose, xylose, lactate, acetate",6
"Medium 8(wheat bran 170, 10)","arabinose, glucose, galactose, xylose, mannose, fructose, lactate",7
"Medium 9(wheat bran 170, 10/ 2x)","arabinose, glucose, galactose, xylose, mannose, fructose, lactate",7
"Medium 10(wheat bran 170, 5)","arabinose, glucose, galactose, xylose, mannose, fructose, lactate",7


### Basal exchange candidates from the supplied models

Start from exchanges permitting uptake in at least one supplied model, then exclude experimental candidate carbon exchanges and oxygen. Include the biotinate exchange identified by the sensitivity test. This separates the model's existing basal candidates from variable inputs without inventing an experimental medium recipe.

The summary reports signed bound ranges across models; `basal_exchange_bounds` retains exact per-model bounds. A lower-bound range of −1000 to 0, for example, means uptake is open in some models and closed in others. Water/proton exchanges can serve balancing roles, and `ash 1 g` is a lumped model component whose physical recipe is unspecified. These are **model basal candidates**, not proven essential nutrients or a confirmed experimental basal medium. Essentiality and a working basal policy require the next functional tests.

In [15]:
variable_exchange_ids = {
    reaction_id for field in carbon_fields for reaction_id in chemical_exchange_ids[field]
} | {oxygen_exchange_id}
supplied_uptake_exchange_ids = set().union(*(set(model.medium) for model in models.values()))
basal_exchange_ids = sorted(
    (supplied_uptake_exchange_ids - variable_exchange_ids) | {biotinate_exchange_id}
)
basal_bound_rows = []
for reaction_id in basal_exchange_ids:
    for key in model_keys:
        if reaction_id in exchanges_by_model[key]:
            reaction = exchanges_by_model[key][reaction_id]
            basal_bound_rows.append({
                "Exchange ID": reaction_id,
                "Model key": key,
                "Substances": ", ".join(m.name or m.id for m in reaction.metabolites),
                "Lower bound": reaction.lower_bound,
                "Upper bound": reaction.upper_bound,
                "Uptake permitted": reaction_id in models[key].medium,
            })
basal_exchange_bounds = pd.DataFrame(basal_bound_rows)
basal_exchange_inventory = basal_exchange_bounds.groupby("Exchange ID", sort=True).agg(
    **{
        "Substances": ("Substances", lambda names: " / ".join(sorted(set(names)))),
        "Models containing exchange": ("Model key", "nunique"),
        "Models permitting uptake": ("Uptake permitted", "sum"),
        "Minimum lower bound": ("Lower bound", "min"),
        "Maximum lower bound": ("Lower bound", "max"),
        "Minimum upper bound": ("Upper bound", "min"),
        "Maximum upper bound": ("Upper bound", "max"),
    }
)
with pd.option_context("display.max_colwidth", None):
    display(basal_exchange_inventory)
print("Exact signed bounds by model are available in basal_exchange_bounds.")

,Substances,Models containing exchange,Models permitting uptake,Minimum lower bound,Maximum lower bound,Minimum upper bound,Maximum upper bound
Exchange ID,,,,,,,
UF02549_E,ash 1 g,12,12,-1000.0,-1000.0,1000.0,1000.0
UF02765_E,hydrogenphosphate,12,12,-1000.0,-1000.0,1000.0,1000.0
UF02997_E,biotinate,12,2,-1000.0,0.0,1000.0,1000.0
UF03268_E,iron(2+),12,12,-1000.0,-1000.0,1000.0,1000.0
UF03314_E,water,12,12,-1000.0,-1000.0,1000.0,1000.0
UF03376_E,ammonium,12,12,-1000.0,-1000.0,1000.0,1000.0
UF03456_E,sulfate,12,12,-1000.0,-1000.0,1000.0,1000.0
UF03474_E,H+,12,12,-1000.0,-1000.0,1000.0,1000.0


Exact signed bounds by model are available in basal_exchange_bounds.


## Individual carbon-source growth support

Replace glucose with each candidate carbon source, one at a time, in all 12 models. Include D-arabinose and separate L- and D-lactate **sensitivity cases** because the experimental CSV does not specify their stereochemistry. Do not open both lactate exchanges in one test.

Use the eight documented basal exchanges, including biotinate, with lower bounds −1000. Set oxygen's lower bound to −1000 for this initial aerobic screen, matching the supplied-model setting. Each tested carbon exchange has lower bound −10; its existing upper bound is retained. These are explicit diagnostic flux bounds, not experimental concentrations or equal-carbon-normalized conditions. Oxygen remains a variable input in the development plan; this screen holds it fixed to isolate carbon-source effects.

Close all other exchange uptake, then open only basal ingredients, oxygen, and the selected carbon source. The **No added carbon** control omits the selected source while retaining basal ingredients, including biotinate and the lumped ash component. It is not chemically carbon-free. Positive control growth therefore requires investigation rather than being silently discarded.

Use ordinary FBA and report actual **signed** carbon/oxygen exchange fluxes; negative values denote uptake for these exchanges. Classify biomass above `screening_growth_tolerance=1e-6` as positive growth for this diagnostic (without changing solver tolerances). Missing exchanges are reported as `missing exchange`, with missing numeric results, rather than zero growth. All modifications are temporary; supplied models and baseline results are retained.

In [16]:
working_basal_lower_bounds = {
    reaction_id: -1000.0 for reaction_id in (
        "UF02549_E", "UF02765_E", "UF02997_E", "UF03268_E",
        "UF03314_E", "UF03376_E", "UF03456_E", "UF03474_E",
    )
}
screening_oxygen_lower_bound = -1000.0
screening_carbon_lower_bound = -10.0
screening_growth_tolerance = 1e-6
carbon_test_cases = [
    ("No added carbon", None, None),
    ("D-arabinose (sensitivity)", "arabinose", "UF03192_E"),
    ("D-glucose", "glucose", "UF03288_E"),
    ("D-galactose", "galactose", "UF03281_E"),
    ("D-xylose", "xylose", "UF03503_E"),
    ("D-mannose", "manose", "UF03359_E"),
    ("D-fructose", "fructose", "UF03275_E"),
    ("L-lactate (sensitivity)", "lactate", "UF02967_E"),
    ("D-lactate (sensitivity)", "lactate", "UF02971_E"),
    ("Acetate", "acetate", "UF03160_E"),
]
carbon_test_rows = []
carbon_test_solutions = {}

for model_key in model_keys:
    model = models[model_key]
    for test_name, csv_field, carbon_exchange_id in carbon_test_cases:
        bounds = {**working_basal_lower_bounds, oxygen_exchange_id: screening_oxygen_lower_bound}
        if carbon_exchange_id is not None:
            bounds[carbon_exchange_id] = screening_carbon_lower_bound
        missing = sorted(set(bounds) - set(exchanges_by_model[model_key]))
        solution = None
        if missing:
            status = "missing exchange"
        else:
            with model:
                model.medium = {}
                for reaction_id, lower_bound in bounds.items():
                    reaction = model.reactions.get_by_id(reaction_id)
                    if len(reaction.metabolites) != 1 or next(iter(reaction.metabolites.values())) != -1:
                        raise ValueError(f"Unexpected uptake direction for {reaction_id} in {model_key}")
                    reaction.lower_bound = lower_bound
                try:
                    solution = model.optimize()
                    status = solution.status
                except OptimizationError:
                    status = model.solver.status or "error"
        optimal = solution is not None and status == "optimal"
        biomass = solution.objective_value if optimal else float("nan")
        carbon_test_solutions[(model_key, test_name)] = solution
        carbon_test_rows.append({
            "Model label": model_labels[model_key],
            "Model key": model_key,
            "Test": test_name,
            "Experimental CSV column": csv_field if csv_field is not None else "Not applicable",
            "Carbon exchange ID": carbon_exchange_id,
            "Solver status": status,
            "Missing exchange IDs": ", ".join(missing),
            "Biomass": biomass,
            "Carbon exchange flux": (
                solution.fluxes[carbon_exchange_id]
                if optimal and carbon_exchange_id is not None else float("nan")
            ),
            "Oxygen exchange flux": solution.fluxes[oxygen_exchange_id] if optimal else float("nan"),
            "Growth outcome": (
                "positive growth" if biomass > screening_growth_tolerance else "no growth"
            ) if optimal else status,
        })

carbon_source_results = pd.DataFrame(carbon_test_rows)
test_order = [test_name for test_name, _, _ in carbon_test_cases]
carbon_source_biomass = carbon_source_results.pivot(
    index="Model label", columns="Test", values="Biomass"
).reindex(columns=test_order)
carbon_source_outcomes = carbon_source_results.pivot(
    index="Model label", columns="Test", values="Growth outcome"
).reindex(columns=test_order)
carbon_source_exchange_fluxes = carbon_source_results.pivot(
    index="Model label", columns="Test", values="Carbon exchange flux"
).reindex(columns=test_order)

print("Biomass under each single-source condition:")
display(carbon_source_biomass)
print("Growth outcomes (missing exchanges are distinct from zero growth):")
with pd.option_context("display.max_colwidth", None):
    display(carbon_source_outcomes)
print("Actual signed carbon exchange fluxes; negative values indicate uptake:")
display(carbon_source_exchange_fluxes)
print("Full results, including oxygen flux and missing IDs, are in carbon_source_results.")
control_growth = carbon_source_results.loc[
    carbon_source_results["Test"].eq("No added carbon")
    & carbon_source_results["Growth outcome"].eq("positive growth")
]
if not control_growth.empty:
    print("Investigate positive growth in the no-added-carbon control:")
    display(control_growth[["Model label", "Model key", "Biomass", "Oxygen exchange flux"]])

Biomass under each single-source condition:


Test,No added carbon,D-arabinose (sensitivity),D-glucose,D-galactose,D-xylose,D-mannose,D-fructose,L-lactate (sensitivity),D-lactate (sensitivity),Acetate
Model label,,,,,,,,,,
M01,0.0,0.0,1.134379,1.134379,0.945316,1.134379,1.134379,0.405907,0.405907,0.0
M02,0.0,0.0,1.143142,NaN,0.000000,1.143142,1.143142,0.000000,0.000000,0.0
M03,0.0,0.0,1.136288,0.000000,0.946907,1.136288,1.136288,0.407191,0.407191,0.0
M04,0.0,0.0,1.028320,0.000000,0.000000,1.028320,1.028320,0.000000,0.000000,0.0
M05,0.0,0.0,0.866986,0.866986,0.722489,0.866986,0.866986,0.000000,0.000000,0.0
M06,0.0,0.0,0.928571,0.928571,0.773809,0.928571,0.928571,0.000000,0.000000,NaN
M07,0.0,0.0,0.873915,0.000000,0.728263,0.873915,0.000000,0.248283,0.248283,0.0
M08,0.0,0.0,1.116792,NaN,0.930660,1.116792,1.116792,0.000000,0.000000,0.0
M09,0.0,0.0,0.887343,NaN,NaN,0.887343,0.887343,0.189203,0.189203,0.0


Growth outcomes (missing exchanges are distinct from zero growth):


Test,No added carbon,D-arabinose (sensitivity),D-glucose,D-galactose,D-xylose,D-mannose,D-fructose,L-lactate (sensitivity),D-lactate (sensitivity),Acetate
Model label,,,,,,,,,,
M01,no growth,no growth,positive growth,positive growth,positive growth,positive growth,positive growth,positive growth,positive growth,no growth
M02,no growth,no growth,positive growth,missing exchange,no growth,positive growth,positive growth,no growth,no growth,no growth
M03,no growth,no growth,positive growth,no growth,positive growth,positive growth,positive growth,positive growth,positive growth,no growth
M04,no growth,no growth,positive growth,no growth,no growth,positive growth,positive growth,no growth,no growth,no growth
M05,no growth,no growth,positive growth,positive growth,positive growth,positive growth,positive growth,no growth,no growth,no growth
M06,no growth,no growth,positive growth,positive growth,positive growth,positive growth,positive growth,no growth,no growth,missing exchange
M07,no growth,no growth,positive growth,no growth,positive growth,positive growth,no growth,positive growth,positive growth,no growth
M08,no growth,no growth,positive growth,missing exchange,positive growth,positive growth,positive growth,no growth,no growth,no growth
M09,no growth,no growth,positive growth,missing exchange,missing exchange,positive growth,positive growth,positive growth,positive growth,no growth


Actual signed carbon exchange fluxes; negative values indicate uptake:


Test,No added carbon,D-arabinose (sensitivity),D-glucose,D-galactose,D-xylose,D-mannose,D-fructose,L-lactate (sensitivity),D-lactate (sensitivity),Acetate
Model label,,,,,,,,,,
M01,NaN,0.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,0.0
M02,NaN,0.0,-10.0,NaN,0.0,-10.0,-10.0,0.0,0.0,0.0
M03,NaN,0.0,-10.0,0.0,-10.0,-10.0,-10.0,-10.0,-10.0,0.0
M04,NaN,0.0,-10.0,0.0,0.0,-10.0,-10.0,0.0,0.0,0.0
M05,NaN,0.0,-10.0,-10.0,-10.0,-10.0,-10.0,0.0,0.0,0.0
M06,NaN,0.0,-10.0,-10.0,-10.0,-10.0,-10.0,0.0,0.0,NaN
M07,NaN,0.0,-10.0,0.0,-10.0,-10.0,0.0,-10.0,-10.0,0.0
M08,NaN,0.0,-10.0,NaN,-10.0,-10.0,-10.0,0.0,0.0,0.0
M09,NaN,0.0,-10.0,NaN,NaN,-10.0,-10.0,-10.0,-10.0,0.0


Full results, including oxygen flux and missing IDs, are in carbon_source_results.
